# * Verify Actual : New Org 2026

In [7]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [8]:
''' Input parameter '''

op_dir = 'data'
op_file = f'uat_new_org_{str_curr_dt}'

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_file: {op_file}')


Parameter input...

   -> op_dir: data
   -> op_file: uat_new_org_20260224


## Import Transaction
-> AGG_PERF_NEWCO

In [9]:
''' Execute transaction '''


# Input parameter
v_start_date = 20260101
v_end_date = 20260131
print(f'\nParameter input...')
print(f'   -> v_start_date: {v_start_date}')
print(f'   -> v_end_date: {v_end_date}')

curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()

query = (f"""
    SELECT /*+PARALLEL(8)*/
        TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME, NATIONWIDE
        , GX2 - NEW_GX2 AS "DIFF_BMA : East"
        , GX3 - NEW_GX3 AS "DIFF_East"
        , GX5 - NEW_GX5 AS "DIFF_Northeast 1"
        , GX6 - NEW_GX6 AS "DIFF_Northeast 2"
        , GX2, GX3, GX5, GX6, NEW_GX2, NEW_GX3, NEW_GX5, NEW_GX6, PPN_TM
    FROM (
        SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME--, AREA_TYPE, AREA_CD, AREA_NAME
            , SUM(CASE WHEN AREA_CD = 'P' THEN ACTUAL_SNAP END) AS NATIONWIDE
            , SUM(CASE WHEN AREA_CD = 'GX2' THEN ACTUAL_SNAP END) AS GX2
            , SUM(CASE WHEN AREA_CD = 'GX3' THEN ACTUAL_SNAP END) AS GX3
            , SUM(CASE WHEN AREA_CD = 'GX5' THEN ACTUAL_SNAP END) AS GX5
            , SUM(CASE WHEN AREA_CD = 'GX6' THEN ACTUAL_SNAP END) AS GX6
            , SUM(CASE WHEN AREA_CD IN ('026', '029', '031', '033', '037', '042', '043', '044', '045', '046', '003', '004') THEN ACTUAL_SNAP END) AS NEW_GX2
            , SUM(CASE WHEN AREA_CD IN ('20A', '20B', '21Z', '23Y', '24X', '26X') THEN ACTUAL_SNAP END) AS NEW_GX3
            , SUM(CASE WHEN AREA_CD IN ('36X', '38X', '39X', '40Z', '48X', '46X') THEN ACTUAL_SNAP END) AS NEW_GX5
            , SUM(CASE WHEN AREA_CD IN ('30X', '31Z', '33X', '34Z', '37X', '44X') THEN ACTUAL_SNAP END) AS NEW_GX6
            , MAX(PPN_TM) PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD IN (
            -->> Revenue
            'DB2R010100' --Postpaid Revenue B2C : DTAC
            , 'TB2R010100' --Postpaid Revenue B2C : TMH
            , 'DB1R000100' --Prepaid Revenue : DTAC
            , 'TB1R000100' --Prepaid Revenue : TMH
            , 'TB3R000100' --TOL Revenue
            , 'TB4R000100' --TVS Revenue
            -->> GA
            , 'DB2S010100CG' --Postpaid Gross Adds B2C : DTAC - GEO Channel
            , 'TB2S010100CG' --Postpaid Gross Adds B2C : TMH - GEO Channel
            , 'DB1S000101CG' --Prepaid Gross Adds : DTAC - GEO Channel
            , 'TB1S000101CG' --Prepaid Gross Adds : TMH - GEO Channel
            , 'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
            , 'TB4S001400CG' --TVS Now Gross Adds - GEO Channel
            -->> M1
            , 'DB2R010500CG' --Postpaid Inflow M1 B2C : DTAC - GEO Channel
            , 'TB2R010500CG' --Postpaid Inflow M1 B2C : TMH - GEO Channel
            , 'DB1R000900CG' --Prepaid Inflow M1 : DTAC - GEO Channel
            , 'TB1R000900CG' --Prepaid Inflow M1 : TMH - GEO Channel
            , 'TB3R000601CG' --TOL Inflow M1 Connected : Consumer - GEO Channel
            , 'TB4R001700CG' --TVS Now Inflow M1 - GEO Channel
            )
        AND AREA_CD IN ('P', 'GX2', 'GX3', 'GX5', 'GX6'
            , '026', '029', '031', '033', '037', '042', '043', '044', '045', '046', '003', '004' --GX2 -> BMA : East
            , '20A', '20B', '21Z', '23Y', '24X', '26X' --GX3 -> East
            , '36X', '38X', '39X', '40Z', '48X', '46X' --GX5 -> Northeast 1
            , '30X', '31Z', '33X', '34Z', '37X', '44X' --GX6 -> Northeast 2
            )
        AND TM_KEY_DAY >= {v_start_date}
        --AND TM_KEY_DAY BETWEEN {v_start_date} AND {v_end_date}
        GROUP BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME
    ) TMP
    WHERE NATIONWIDE IS NOT NULL

    UNION ALL

    SELECT 
        TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME, NATIONWIDE
        , GX2 - NEW_GX2 AS "DIFF_BMA : East"
        , GX3 - NEW_GX3 AS "DIFF_East"
        , GX5 - NEW_GX5 AS "DIFF_Northeast 1"
        , GX6 - NEW_GX6 AS "DIFF_Northeast 2"
        , GX2, GX3, GX5, GX6, NEW_GX2, NEW_GX3, NEW_GX5, NEW_GX6, PPN_TM
    FROM (
        SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME--, AREA_TYPE, AREA_CD, AREA_NAME
            , SUM(CASE WHEN AREA_CD = 'P' THEN ACTUAL_AGG_MTH END) AS NATIONWIDE
            , SUM(CASE WHEN AREA_CD = 'GX2' THEN ACTUAL_AGG_MTH END) AS GX2
            , SUM(CASE WHEN AREA_CD = 'GX3' THEN ACTUAL_AGG_MTH END) AS GX3
            , SUM(CASE WHEN AREA_CD = 'GX5' THEN ACTUAL_AGG_MTH END) AS GX5
            , SUM(CASE WHEN AREA_CD = 'GX6' THEN ACTUAL_AGG_MTH END) AS GX6
            , SUM(CASE WHEN AREA_CD IN ('026', '029', '031', '033', '037', '042', '043', '044', '045', '046', '003', '004') THEN ACTUAL_AGG_MTH END) AS NEW_GX2
            , SUM(CASE WHEN AREA_CD IN ('20A', '20B', '21Z', '23Y', '24X', '26X') THEN ACTUAL_AGG_MTH END) AS NEW_GX3
            , SUM(CASE WHEN AREA_CD IN ('36X', '38X', '39X', '40Z', '48X', '46X') THEN ACTUAL_AGG_MTH END) AS NEW_GX5
            , SUM(CASE WHEN AREA_CD IN ('30X', '31Z', '33X', '34Z', '37X', '44X') THEN ACTUAL_AGG_MTH END) AS NEW_GX6
            , MAX(PPN_TM) PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD IN (
            -->> Subs
            'DB2S010600' --Postpaid Reported SubBase B2C : DTAC
            , 'TB2S010600' --Postpaid Reported SubBase B2C : TMH
            , 'DB1S000600' --Prepaid Active Caller 30D Rolling : DTAC
            , 'TB1S000600' --Prepaid Active Caller 30D Rolling : TMH
            , 'DB1S000600' --Prepaid Active Caller 30D Rolling : DTAC
            , 'DB1S000601' --Prepaid Active Caller 30D Rolling : Thai Mass : DTAC
            , 'DB1S000602' --Prepaid Active Caller 30D Rolling : AEC/Migrants : DTAC
            , 'DB1S000603' --Prepaid Active Caller 30D Rolling : Tourists : DTAC
            , 'TB1S000600' --Prepaid Active Caller 30D Rolling : TMH
            , 'TB1S000601' --Prepaid Active Caller 30D Rolling : Thai Mass : TMH
            , 'TB1S000602' --Prepaid Active Caller 30D Rolling : AEC/Migrants : TMH
            , 'TB1S000603' --Prepaid Active Caller 30D Rolling : Tourists : TMH
            , 'TB3S000600' --FTTx Reported SubBase
            , 'TB4S000500' --TVS Active Subs
            --, 'TNSC00150' --TrueID TV Box MAU
            )
        AND AREA_CD IN ('P', 'GX2', 'GX3', 'GX5', 'GX6'
            , '026', '029', '031', '033', '037', '042', '043', '044', '045', '046', '003', '004' --GX2 -> BMA : East
            , '20A', '20B', '21Z', '23Y', '24X', '26X' --GX3 -> East
            , '36X', '38X', '39X', '40Z', '48X', '46X' --GX5 -> Northeast 1
            , '30X', '31Z', '33X', '34Z', '37X', '44X' --GX6 -> Northeast 2
            )
        AND TM_KEY_DAY >= {v_start_date}
        AND TM_KEY_DAY IN (20250131, 20250229, 20250331, 20250430, 20250531, 20250630, 20250731, 20250831, 20250930, 20251031, 20251130, 20251231
            , 20260131, 20260228, 20260331, 20260430, 20260531, 20260630, 20260731, 20260831, 20260930, 20261031, 20261130, 20261231)
        GROUP BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD, METRIC_NAME
    ) TMP
    WHERE NATIONWIDE IS NOT NULL

    --ORDER BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_GRP, COMP_CD, METRIC_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')

    # # Generate CSV file
    # src_df.to_csv(f'{op_dir}/{op_vis_file}.csv', index=False, encoding='utf-8')
    # print(f'\n   -> Generate "{op_vis_file}.csv" successfully')

    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Parameter input...
   -> v_start_date: 20260101
   -> v_end_date: 20260131

Data as of 2026-02-24, 11:04:50



DataFrame: 786 rows, 21 columns


## Overview

In [10]:
# src_df.info()

In [11]:
# src_df.head(3)

In [12]:
x_df = src_df.iloc[:, :12].query(" TM_KEY_MTH >= 202601 ").copy()
# x_df = src_df.iloc[:, :12].query(" TM_KEY_MTH >= 202501 ").copy()
# x_df = src_df.iloc[:, :12].query(" TM_KEY_MTH == 202501 ").copy()
x_df['Org_Diff'] = x_df['DIFF_BMA : East'] + x_df['DIFF_East'] + x_df['DIFF_Northeast 1'] + x_df['DIFF_Northeast 2']

# diff_summary_df = x_df.query(" Org_Diff != 0 ").reset_index(drop=True)
diff_summary_df = x_df.query(" Org_Diff > 0 or Org_Diff < 0 ").reset_index(drop=True)
diff_summary_df = diff_summary_df.groupby(['TM_KEY_MTH','METRIC_GRP','PRODUCT_GRP','COMP_CD','METRIC_CD','METRIC_NAME']).agg({'NATIONWIDE':'sum', 'Org_Diff':'sum'})

cols_to_fix = ['NATIONWIDE', 'Org_Diff']
for col in cols_to_fix:
    diff_summary_df[col] = diff_summary_df[col].apply(lambda x: format(x, ',.0f'))
    
diff_summary_df

NATIONWIDE  \
TM_KEY_MTH METRIC_GRP PRODUCT_GRP COMP_CD METRIC_CD    METRIC_NAME                                       
202601     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC       2,286,276,647   
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC            1,843,535,346   
                      TOL         TRUE    TB3R000100   TOL Revenue                       1,660,956,221   
                      TVS         TRUE    TB4R000100   TVS Revenue                         246,518,898   
           Sales      TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel          69,604   
                                          TB4S001400CG TVS Now Gross Adds - GEO Channel            273   
202602     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC       1,543,514,453   
                                  TRUE    TB2R010100   Postpaid Revenue B2C : TMH        2,363,797,630   
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC            1,270,272,794   
                                  TRUE    TB1R000100   Prepaid Revenue : TMH                69,500,304   
                      TOL         TRUE    TB3R000100   TOL Revenue                       1,154,201,036   
                      TVS         TRUE    TB4R000100   TVS Revenue                         113,461,432   

                                                                                          Org_Diff  
TM_KEY_MTH METRIC_GRP PRODUCT_GRP COMP_CD METRIC_CD    METRIC_NAME                                  
202601     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC           9,109  
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC              113,826  
                      TOL         TRUE    TB3R000100   TOL Revenue                               0  
                      TVS         TRUE    TB4R000100   TVS Revenue                        -739,735  
           Sales      TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel      -1,305  
                                          TB4S001400CG TVS Now Gross Adds - GEO Channel         -4  
202602     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC           4,228  
                                  TRUE    TB2R010100   Postpaid Revenue B2C : TMH         -635,951  
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC               20,650  
                                  TRUE    TB1R000100   Prepaid Revenue : TMH                    -7  
                      TOL         TRUE    TB3R000100   TOL Revenue                       3,634,182  
                      TVS         TRUE    TB4R000100   TVS Revenue                        -231,857

In [13]:
all_summary_df = x_df.groupby(['TM_KEY_MTH','METRIC_GRP','PRODUCT_GRP','COMP_CD','METRIC_CD','METRIC_NAME']).agg({'NATIONWIDE':'sum', 'Org_Diff':'sum'})

cols_to_fix = ['NATIONWIDE', 'Org_Diff']
for col in cols_to_fix:
    all_summary_df[col] = all_summary_df[col].apply(lambda x: format(x, ',.0f'))
    
all_summary_df

NATIONWIDE  \
TM_KEY_MTH METRIC_GRP PRODUCT_GRP COMP_CD METRIC_CD    METRIC_NAME                                                         
202601     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC                         2,362,683,949   
                                  TRUE    TB2R010100   Postpaid Revenue B2C : TMH                          3,410,570,124   
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC                              1,843,535,346   
                                  TRUE    TB1R000100   Prepaid Revenue : TMH                               2,618,975,308   
                      TOL         TRUE    TB3R000100   TOL Revenue                                         1,660,956,221   
                      TVS         TRUE    TB4R000100   TVS Revenue                                           246,518,898   
           Sales      Postpaid    DTAC    DB2R010500CG Postpaid Inflow M1 B2C : DTAC - GEO Channel             7,002,914   
                                          DB2S010100CG Postpaid Gross Adds B2C : DTAC - GEO Channel               13,036   
                                  TRUE    TB2R010500CG Postpaid Inflow M1 B2C : TMH - GEO Channel             30,553,335   
                                          TB2S010100CG Postpaid Gross Adds B2C : TMH - GEO Channel                57,223   
                      Prepaid     DTAC    DB1R000900CG Prepaid Inflow M1 : DTAC - GEO Channel                141,542,806   
                                  TRUE    TB1R000900CG Prepaid Inflow M1 : TMH - GEO Channel                 236,039,111   
                      TOL         TRUE    TB3R000601CG TOL Inflow M1 Connected : Consumer - GEO Channel       21,185,059   
                                          TB3S000102CG TOL Gross Adds Connected : Consumer - GEO Channel          37,801   
                      TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel                           548,803   
                                          TB4S001400CG TVS Now Gross Adds - GEO Channel                            2,642   
           Subs       Postpaid    DTAC    DB2S010600   Postpaid Reported SubBase B2C : DTAC                    4,840,930   
                                  TRUE    TB2S010600   Postpaid Reported SubBase B2C : TMH                     9,083,044   
                      Prepaid     DTAC    DB1S000600   Prepaid Active Caller 30D Rolling : DTAC                9,840,922   
                                          DB1S000601   Prepaid Active Caller 30D Rolling : Thai Mass :...      6,810,674   
                                          DB1S000602   Prepaid Active Caller 30D Rolling : AEC/Migrant...      2,677,652   
                                          DB1S000603   Prepaid Active Caller 30D Rolling : Tourists : ...        352,596   
                                  TRUE    TB1S000600   Prepaid Active Caller 30D Rolling : TMH                16,151,279   
                                          TB1S000601   Prepaid Active Caller 30D Rolling : Thai Mass :...     14,321,837   
                                          TB1S000602   Prepaid Active Caller 30D Rolling : AEC/Migrant...      1,168,880   
                                          TB1S000603   Prepaid Active Caller 30D Rolling : Tourists : TMH        660,562   
                      TOL         TRUE    TB3S000600   FTTx Reported SubBase                                   3,268,512   
                      TVS         TRUE    TB4S000500   TVS Active Subs                                         1,058,669   
202602     Revenue    Postpaid    DTAC    DB2R010100   Postpaid Revenue B2C : DTAC                         1,617,757,930   
                                  TRUE    TB2R010100   Postpaid Revenue B2C : TMH                          2,363,797,630   
                      Prepaid     DTAC    DB1R000100   Prepaid Revenue : DTAC                              1,270,272,794   
                  

## Product detail

In [14]:
revenue_df = x_df.query(" METRIC_GRP=='Revenue' ").reset_index(drop=True)
revenue_summary_df = revenue_df.groupby(['TM_KEY_MTH','PRODUCT_GRP','COMP_CD','METRIC_CD','METRIC_NAME']).agg({'NATIONWIDE':'sum', 'Org_Diff':'sum'})

cols_to_fix = ['NATIONWIDE', 'Org_Diff']
for col in cols_to_fix:
    revenue_summary_df[col] = revenue_summary_df[col].apply(lambda x: format(x, ',.0f'))

revenue_summary_df

NATIONWIDE  \
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD  METRIC_NAME                                  
202601     Postpaid    DTAC    DB2R010100 Postpaid Revenue B2C : DTAC  2,362,683,949   
                       TRUE    TB2R010100 Postpaid Revenue B2C : TMH   3,410,570,124   
           Prepaid     DTAC    DB1R000100 Prepaid Revenue : DTAC       1,843,535,346   
                       TRUE    TB1R000100 Prepaid Revenue : TMH        2,618,975,308   
           TOL         TRUE    TB3R000100 TOL Revenue                  1,660,956,221   
           TVS         TRUE    TB4R000100 TVS Revenue                    246,518,898   
202602     Postpaid    DTAC    DB2R010100 Postpaid Revenue B2C : DTAC  1,617,757,930   
                       TRUE    TB2R010100 Postpaid Revenue B2C : TMH   2,363,797,630   
           Prepaid     DTAC    DB1R000100 Prepaid Revenue : DTAC       1,270,272,794   
                       TRUE    TB1R000100 Prepaid Revenue : TMH        1,711,480,793   
           TOL         TRUE    TB3R000100 TOL Revenue                  1,154,201,036   
           TVS         TRUE    TB4R000100 TVS Revenue                    113,461,432   

                                                                        Org_Diff  
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD  METRIC_NAME                             
202601     Postpaid    DTAC    DB2R010100 Postpaid Revenue B2C : DTAC      9,109  
                       TRUE    TB2R010100 Postpaid Revenue B2C : TMH           0  
           Prepaid     DTAC    DB1R000100 Prepaid Revenue : DTAC         113,826  
                       TRUE    TB1R000100 Prepaid Revenue : TMH                0  
           TOL         TRUE    TB3R000100 TOL Revenue                          0  
           TVS         TRUE    TB4R000100 TVS Revenue                   -739,735  
202602     Postpaid    DTAC    DB2R010100 Postpaid Revenue B2C : DTAC      4,228  
                       TRUE    TB2R010100 Postpaid Revenue B2C : TMH    -635,951  
           Prepaid     DTAC    DB1R000100 Prepaid Revenue : DTAC          20,650  
                       TRUE    TB1R000100 Prepaid Revenue : TMH               -7  
           TOL         TRUE    TB3R000100 TOL Revenue                  3,634,182  
           TVS         TRUE    TB4R000100 TVS Revenue                   -231,857

In [15]:
sale_df = x_df.query(" METRIC_GRP=='Sales' ").reset_index(drop=True)
sale_summary_df = sale_df.groupby(['TM_KEY_MTH','PRODUCT_GRP','COMP_CD','METRIC_CD','METRIC_NAME']).agg({'NATIONWIDE':'sum', 'Org_Diff':'sum'})

cols_to_fix = ['NATIONWIDE', 'Org_Diff']
for col in cols_to_fix:
    sale_summary_df[col] = sale_summary_df[col].apply(lambda x: format(x, ',.0f'))

sale_summary_df

NATIONWIDE  \
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD    METRIC_NAME                                                      
202601     Postpaid    DTAC    DB2R010500CG Postpaid Inflow M1 B2C : DTAC - GEO Channel          7,002,914   
                               DB2S010100CG Postpaid Gross Adds B2C : DTAC - GEO Channel            13,036   
                       TRUE    TB2R010500CG Postpaid Inflow M1 B2C : TMH - GEO Channel          30,553,335   
                               TB2S010100CG Postpaid Gross Adds B2C : TMH - GEO Channel             57,223   
           Prepaid     DTAC    DB1R000900CG Prepaid Inflow M1 : DTAC - GEO Channel             141,542,806   
                       TRUE    TB1R000900CG Prepaid Inflow M1 : TMH - GEO Channel              236,039,111   
           TOL         TRUE    TB3R000601CG TOL Inflow M1 Connected : Consumer - GEO Channel    21,185,059   
                               TB3S000102CG TOL Gross Adds Connected : Consumer - GEO Channel       37,801   
           TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel                        548,803   
                               TB4S001400CG TVS Now Gross Adds - GEO Channel                         2,642   
202602     Postpaid    DTAC    DB2R010500CG Postpaid Inflow M1 B2C : DTAC - GEO Channel          4,151,262   
                               DB2S010100CG Postpaid Gross Adds B2C : DTAC - GEO Channel             7,612   
                       TRUE    TB2R010500CG Postpaid Inflow M1 B2C : TMH - GEO Channel          19,488,989   
                               TB2S010100CG Postpaid Gross Adds B2C : TMH - GEO Channel             36,589   
           Prepaid     DTAC    DB1R000900CG Prepaid Inflow M1 : DTAC - GEO Channel              72,291,970   
                       TRUE    TB1R000900CG Prepaid Inflow M1 : TMH - GEO Channel              116,054,195   
           TOL         TRUE    TB3R000601CG TOL Inflow M1 Connected : Consumer - GEO Channel    12,668,358   
                               TB3S000102CG TOL Gross Adds Connected : Consumer - GEO Channel       22,649   
           TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel                        234,574   
                               TB4S001400CG TVS Now Gross Adds - GEO Channel                         1,135   

                                                                                              Org_Diff  
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD    METRIC_NAME                                                 
202601     Postpaid    DTAC    DB2R010500CG Postpaid Inflow M1 B2C : DTAC - GEO Channel              0  
                               DB2S010100CG Postpaid Gross Adds B2C : DTAC - GEO Channel             0  
                       TRUE    TB2R010500CG Postpaid Inflow M1 B2C : TMH - GEO Channel               0  
                               TB2S010100CG Postpaid Gross Adds B2C : TMH - GEO Channel              0  
           Prepaid     DTAC    DB1R000900CG Prepaid Inflow M1 : DTAC - GEO Channel                   0  
                       TRUE    TB1R000900CG Prepaid Inflow M1 : TMH - GEO Channel                    0  
           TOL         TRUE    TB3R000601CG TOL Inflow M1 Connected : Consumer - GEO Channel         0  
                               TB3S000102CG TOL Gross Adds Connected : Consumer - GEO Channel        0  
           TVS         TRUE    TB4R001700CG TVS Now Inflow M1 - GEO Channel                     -1,305  
                               TB4S001400CG TVS Now Gross Adds - GEO Channel                        -4  
202602     Postpaid    DTAC    DB2R010500CG Postpaid Inflow M1 B2C : DTAC - GEO Channel              0  
                               DB2S010100CG Postpaid Gross Adds B2C : DTAC - GEO Channel             0  
                       TRUE    TB2R010500CG Postpaid Inflow M1 B2C : TMH - GEO Channel               0  
                               TB2S010100CG Postpaid Gross Adds B2C : TMH - GEO Channel             

In [16]:
subs_df = x_df.query(" METRIC_GRP=='Subs' ").reset_index(drop=True)
subs_summary_df = subs_df.groupby(['TM_KEY_MTH','PRODUCT_GRP','COMP_CD','METRIC_CD','METRIC_NAME']).agg({'NATIONWIDE':'sum', 'Org_Diff':'sum'})

cols_to_fix = ['NATIONWIDE', 'Org_Diff']
for col in cols_to_fix:
    subs_summary_df[col] = subs_summary_df[col].apply(lambda x: format(x, ',.0f'))

subs_summary_df

NATIONWIDE  \
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD  METRIC_NAME                                                      
202601     Postpaid    DTAC    DB2S010600 Postpaid Reported SubBase B2C : DTAC                 4,840,930   
                       TRUE    TB2S010600 Postpaid Reported SubBase B2C : TMH                  9,083,044   
           Prepaid     DTAC    DB1S000600 Prepaid Active Caller 30D Rolling : DTAC             9,840,922   
                               DB1S000601 Prepaid Active Caller 30D Rolling : Thai Mass :...   6,810,674   
                               DB1S000602 Prepaid Active Caller 30D Rolling : AEC/Migrant...   2,677,652   
                               DB1S000603 Prepaid Active Caller 30D Rolling : Tourists : ...     352,596   
                       TRUE    TB1S000600 Prepaid Active Caller 30D Rolling : TMH             16,151,279   
                               TB1S000601 Prepaid Active Caller 30D Rolling : Thai Mass :...  14,321,837   
                               TB1S000602 Prepaid Active Caller 30D Rolling : AEC/Migrant...   1,168,880   
                               TB1S000603 Prepaid Active Caller 30D Rolling : Tourists : TMH     660,562   
           TOL         TRUE    TB3S000600 FTTx Reported SubBase                                3,268,512   
           TVS         TRUE    TB4S000500 TVS Active Subs                                      1,058,669   

                                                                                             Org_Diff  
TM_KEY_MTH PRODUCT_GRP COMP_CD METRIC_CD  METRIC_NAME                                                  
202601     Postpaid    DTAC    DB2S010600 Postpaid Reported SubBase B2C : DTAC                      0  
                       TRUE    TB2S010600 Postpaid Reported SubBase B2C : TMH                       0  
           Prepaid     DTAC    DB1S000600 Prepaid Active Caller 30D Rolling : DTAC                  0  
                               DB1S000601 Prepaid Active Caller 30D Rolling : Thai Mass :...        0  
                               DB1S000602 Prepaid Active Caller 30D Rolling : AEC/Migrant...        0  
                               DB1S000603 Prepaid Active Caller 30D Rolling : Tourists : ...        0  
                       TRUE    TB1S000600 Prepaid Active Caller 30D Rolling : TMH                   0  
                               TB1S000601 Prepaid Active Caller 30D Rolling : Thai Mass :...        0  
                               TB1S000602 Prepaid Active Caller 30D Rolling : AEC/Migrant...        0  
                               TB1S000603 Prepaid Active Caller 30D Rolling : Tourists : TMH        0  
           TOL         TRUE    TB3S000600 FTTx Reported SubBase                                     0  
           TVS         TRUE    TB4S000500 TVS Active Subs                                           0